# Exp127 v2 - FINE-TUNE the incumbent, with learning curves

**No submission, no slot.**

## Why v1 was wrong

v1 trained from scratch. Exp125 then read the incumbent checkpoint and settled a
question we had been guessing at:

    checkpoint_last.pth -> epoch: 402   best_score: 0.9835   fold: 0
                           optimizer_state_dict: PRESENT

The pack's `50ep` name is misleading - it is a **402-epoch** model. A from-scratch
run fits about `30,900` iterations in a 9h kernel (measured: 1.02 s/iter at batch
4, plus 397 s startup), which cannot approach 402 full epochs. v1 would have
produced a weak model and so a weak ensemble member.

Because `optimizer_state_dict` is present, the same GPU hours can instead
**continue training from epoch 402**.

## What this run does

Resumes the full model - `TemporalUNet3D` + `detect_head` + `SimpleNodeTransformer` -
and its AdamW state, then continues on **fold 1**, a different 180/19 partition,
at a reduced learning rate.

The script has no resume path (`--unet-weights` loads the UNet *only*, not the
transformer or the optimizer), so we patch the copy in `/kaggle/working` and
**assert each edit landed**. A silently-skipped patch would quietly train from
scratch while looking fine - exactly the failure we already hit once with the TTA
substring patch.

## Honest caveat

Fold 0 - the incumbent's training set - covers most of fold 1's validation
movies. So validation numbers here are **optimistic**, and the fine-tuned model
stays **correlated** with the incumbent, limiting ensemble gain.

This yields a stronger *single* model. A genuinely independent member would need
a from-scratch run, which we now know is too weak to justify the hours. Judge
this on the leaderboard, not on the validation curve.


In [ ]:
try:
    import zarr, geff, tracksdata
    
except: 
    import os
    import sys
    import subprocess
    import importlib
    import importlib.util
    from pathlib import Path
    
    # Polars wheel in the support package uses the 32-bit runtime package.
    os.environ.setdefault("POLARS_PREFER_PKG", "32")
    
    SUPPORT_DIR = Path(
        "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1"
    )
    WHEELS_DIR = SUPPORT_DIR / "wheels"
    
    if not WHEELS_DIR.exists():
        # Fallback for Kaggle's shorter mounted dataset path.
        candidates = list(Path("/kaggle/input").glob("**/wheels"))
        if not candidates:
            raise FileNotFoundError("Could not find the attached offline wheels directory")
        WHEELS_DIR = candidates[0]
    
    print("Offline wheels:", WHEELS_DIR)
    
    
    # These are packages supplied by the attached dataset.
    # Deliberately exclude:
    #   numpy, scipy, torch, pandas, dask, xarray, numba, llvmlite
    #
    # Kaggle already supplies compatible versions of those packages.
    OFFLINE_PACKAGES = [
        "tracksdata",
        "zarr==3.2.1",
        "numcodecs==0.15.1",
        "donfig==0.8.1.post1",
        "geff==1.2.0.1.1",
        "geff-spec==1.1.1",
        "pyscipopt==6.2.1",
        "ilpy==0.6.0",
        "rustworkx==0.18.0",
        "polars==1.42.0",
        "polars-runtime-32==1.42.0",
        "bidict==0.23.1",
        "imagecodecs==2026.6.26",
    ]
    
    
    def module_missing(module_name: str) -> bool:
        return importlib.util.find_spec(module_name) is None
    
    
    # Import name differs from pip package name in several cases.
    REQUIRED_IMPORTS = {
        "tracksdata": "tracksdata",
        "zarr": "zarr",
        "numcodecs": "numcodecs",
        "geff": "geff",
        "pyscipopt": "pyscipopt",
        "ilpy": "ilpy",
        "rustworkx": "rustworkx",
        "polars": "polars",
        "imagecodecs": "imagecodecs",
    }
    
    
    def purge_modules(module_roots):
        """
        Remove already-imported package modules from sys.modules.
    
        Normally this cell runs before imports, but this also protects against
        accidental imports performed by earlier Kaggle initialization code.
        """
        for root in module_roots:
            for name in list(sys.modules):
                if name == root or name.startswith(root + "."):
                    sys.modules.pop(name, None)
    
    
    def install_offline_packages():
        cmd = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-index",
            "--no-deps",
            "--find-links",
            str(WHEELS_DIR),
            *OFFLINE_PACKAGES,
        ]
    
        print("Installing attached packages without modifying NumPy/SciPy/Torch...")
        result = subprocess.run(
            cmd,
            text=True,
            capture_output=True,
        )
    
        if result.returncode != 0:
            print(result.stdout[-4000:])
            print(result.stderr[-4000:])
            raise RuntimeError("Offline dependency installation failed")
    
        purge_modules(REQUIRED_IMPORTS.values())
    
    
    install_offline_packages()
    
    
    # Verify the complete import chain needed by biohub_tracking.io.
    failures = {}
    
    for name, module_name in {
        **REQUIRED_IMPORTS,
        "numpy": "numpy",
        "scipy": "scipy",
        "dask": "dask.array",
        "xarray": "xarray",
    }.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    
    if failures:
        raise ImportError(
            "Dependency verification failed:\n"
            + "\n".join(f"{name}: {error}" for name, error in failures.items())
        )

import zarr, geff, tracksdata
print("All offline dependencies imported successfully.")

In [ ]:
# ---- configuration -------------------------------------------------------
FOLD          = 1        # different partition from the incumbent's fold 0
BATCH         = 4
MAX_ITERS     = 150      # batches per "epoch"; NOT comparable to a full-dataset epoch
EPOCHS        = 400      # upper bound; the wall-clock guard stops us first
TARGET_HOURS  = 7.5      # headroom inside the 9h limit
LR            = 2e-5     # REDUCED from 1e-4: continuing a converged 402-epoch model
# --------------------------------------------------------------------------
import os, sys, glob, json, time, shutil, subprocess, re, random
from pathlib import Path
import torch

_m=[]
for pat in ("/kaggle/input/*/*/*/src/tracking_cellmot/metrics.py",
            "/kaggle/input/*/*/src/tracking_cellmot/metrics.py",
            "/kaggle/input/**/tracking_cellmot/metrics.py"):
    _m+=glob.glob(pat,recursive=True)
assert _m, "official scorer dataset not attached"
REPO_RO=Path(_m[0]).parents[2]

REPO=Path("/kaggle/working/repo")
if REPO.exists(): shutil.rmtree(REPO)
shutil.copytree(REPO_RO, REPO)
SRC=REPO/"src"; (REPO/"weights").mkdir(exist_ok=True)

hits=glob.glob("/kaggle/input/**/checkpoint_last.pth", recursive=True)
assert hits, "checkpoint_last.pth not found - attach the support pack or weights mirror"
CKPT=hits[0]
_c=torch.load(CKPT, map_location="cpu", weights_only=False)
print(f"resume checkpoint : {CKPT}")
print(f"  epoch      : {_c.get('epoch')}")
print(f"  best_score : {_c.get('best_score')}")
print(f"  fold       : {_c.get('fold')}")
print(f"  has optim  : {'optimizer_state_dict' in _c}")
del _c

COMP=Path("/kaggle/input/competitions/biohub-cell-tracking-during-development")
if not COMP.exists(): COMP=Path(glob.glob("/kaggle/input/*biohub-cell-tracking*")[0])
TRAIN=COMP/"train"
stems=sorted(p.name[:-5] for p in TRAIN.glob("*.zarr") if (TRAIN/f"{p.name[:-5]}.geff").exists())
print(f"\nlabelled movies : {len(stems)}")
print("GPU             :", os.popen("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").read().strip())

SPLITS=Path("/kaggle/working/dataset_splits.json")
folds=[]
for f in range(3):
    s=list(stems); random.Random(100+f).shuffle(s)
    n_val=max(1,len(s)//10)
    folds.append({"split":f,"train":s[n_val:],"test":s[:n_val]})
SPLITS.write_text(json.dumps(folds,indent=1))
print(f"fold {FOLD}          : {len(folds[FOLD]['train'])} train / {len(folds[FOLD]['test'])} val")


In [ ]:
# ---- patch the copied script to support a real resume ----------------------
# --unet-weights loads only the UNet; we also need detect_head, the transformer,
# and the AdamW state. Patch the writable copy and ASSERT each edit landed.
tp = REPO/"scripts"/"train_unet_transformer.py"
src = tp.read_text()

A_MODEL = "    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)"
B_MODEL = "\n".join([
 '    _resume = os.environ.get("BIOHUB_RESUME_CKPT")',
 '    if _resume:',
 '        _ck = torch.load(_resume, map_location="cpu", weights_only=False)',
 '        _sd = _ck.get("model_state_dict", _ck) if isinstance(_ck, dict) else _ck',
 '        _sd = {k.replace("unet.module.", "unet.", 1): v for k, v in _sd.items()}',
 '        _missing, _unexpected = model.load_state_dict(_sd, strict=False)',
 '        print("RESUME: loaded full model from " + str(_resume), flush=True)',
 '        print("RESUME: epoch=%s best=%s missing=%d unexpected=%d" % (',
 '              _ck.get("epoch"), _ck.get("best_score"), len(_missing), len(_unexpected)), flush=True)',
 '        if len(_missing) > 8:',
 '            raise RuntimeError("RESUME: too many missing keys (%d)" % len(_missing))',
 '',
 A_MODEL])
assert src.count(A_MODEL) == 1, "model anchor count %d" % src.count(A_MODEL)
src = src.replace(A_MODEL, B_MODEL, 1)

A_OPT = '    print(f"Starting training for {n_epochs} epochs (batch_size={batch_size})...", flush=True)'
B_OPT = "\n".join([
 '    if os.environ.get("BIOHUB_RESUME_CKPT") and os.environ.get("BIOHUB_RESUME_OPTIM", "1") != "0":',
 '        _ck = torch.load(os.environ["BIOHUB_RESUME_CKPT"], map_location="cpu", weights_only=False)',
 '        if isinstance(_ck, dict) and "optimizer_state_dict" in _ck:',
 '            try:',
 '                optimizer.load_state_dict(_ck["optimizer_state_dict"])',
 '                for g in optimizer.param_groups:',
 '                    g["lr"] = lr',
 '                print("RESUME: optimizer state restored, lr forced to %s" % lr, flush=True)',
 '            except Exception as _e:',
 '                print("RESUME: optimizer NOT restored (%s) - fresh AdamW" % _e, flush=True)',
 '        del _ck',
 '',
 A_OPT])
assert src.count(A_OPT) == 1, "optimizer anchor count %d" % src.count(A_OPT)
src = src.replace(A_OPT, B_OPT, 1)

tp.write_text(src)
import ast as _ast; _ast.parse(src)
t = tp.read_text()
assert "RESUME: loaded full model" in t and "RESUME: optimizer state restored" in t
print("patch applied and verified (model + optimizer resume; script still parses)")


In [ ]:
EPOCH_RE = re.compile(
    r"Epoch\s+(\d+)/\d+\s*\|\s*edge=([\d.]+)\s*\|\s*det=([\d.]+)\s*\|"
    r"\s*test_loss=([\d.]+)\s*\|\s*acc=([\d.]+)\s*\|\s*recall=([\d.]+)\s*\|"
    r"\s*best=([\d.]+)\s*(\*?)\s*\|\s*train=([\d.]+)s\s*test=([\d.]+)s")

env=dict(os.environ); env["PYTHONPATH"]=str(SRC)
env["BIOHUB_RESUME_CKPT"]=str(CKPT)
env["BIOHUB_RESUME_OPTIM"]="1"
cmd=[sys.executable,"-u",str(REPO/"scripts"/"train_unet_transformer.py"),
     "--data-dir",str(TRAIN),"--splits",str(SPLITS),"--split",str(FOLD),
     "--epochs",str(EPOCHS),"--max-iters",str(MAX_ITERS),
     "--batch-size",str(BATCH),"--lr",str(LR),"--num-workers","2","--single-gpu"]
print(" ".join(cmd),"\n",flush=True)

rows=[]; t_start=time.time(); budget=TARGET_HOURS*3600; stopped=None
proc=subprocess.Popen(cmd,cwd=str(REPO),env=env,stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in proc.stdout:
    m=EPOCH_RE.search(line)
    if not m:
        s=line.rstrip()
        if s and not s.startswith(("  iters","Training")): print(s[:160],flush=True)
        continue
    ep,edge,det,tl,acc,rec,best,star,tt,vt=m.groups()
    rows.append(dict(epoch=int(ep),train_edge_loss=float(edge),train_det_loss=float(det),
                     val_loss=float(tl),val_acc=float(acc),val_recall=float(rec),
                     best=float(best),improved=bool(star),train_s=float(tt),val_s=float(vt)))
    el=time.time()-t_start
    print(f"  ep {ep:>3} | train_edge {edge} | val_loss {tl} | acc {acc} | recall {rec} "
          f"| {'*' if star else ' '} | {el/60:.1f} min elapsed",flush=True)
    nxt=rows[-1]["train_s"]+rows[-1]["val_s"]
    if el+nxt*1.15>budget:
        stopped=f"wall-clock guard at epoch {ep} ({el/3600:.2f}h of {TARGET_HOURS}h)"
        print(f"\n{stopped} - terminating; best checkpoint already saved",flush=True)
        proc.terminate(); break
try: proc.wait(timeout=120)
except Exception: proc.kill()
print(f"\ncollected {len(rows)} epochs in {(time.time()-t_start)/60:.1f} min"
      + (f"; stopped by {stopped}" if stopped else ""))


In [ ]:
import pandas as pd
df=pd.DataFrame(rows)
assert len(df), "no epochs parsed - check the log above"
df.to_csv("/kaggle/working/exp127_training_metrics.csv",index=False)

# Table view: the palette's magenta sits below 3:1 contrast on a light surface,
# so the numbers are shown explicitly rather than read from colour alone.
show=df[["epoch","train_edge_loss","train_det_loss","val_loss","val_acc","val_recall","best","train_s"]]
print("first 5 / last 10 epochs:\n")
print(show.head(5).to_string(index=False))
print("  ...")
print(show.tail(10).to_string(index=False))
best_ep=int(df.loc[df.best.idxmax(),"epoch"])
print(f"\nbest epoch: {best_ep}  (score acc*recall = {df.best.max():.4f})")


## Learning curves

Three separate panels, deliberately **not** a dual-axis chart: losses and
rates have different scales, and overlaying them on twin y-axes is the single
most misleading thing a training plot can do. Same-unit series share an axis;
different units get their own panel.

In [ ]:
import matplotlib.pyplot as plt

# Categorical slots 1-3 from a CVD-validated palette, assigned in fixed order.
BLUE, GREEN, MAGENTA = "#2a78d6", "#008300", "#e87ba4"
INK, MUTED = "#1a1a19", "#6b6a63"

def style(ax, title, ylabel):
    ax.set_title(title, fontsize=11, color=INK, loc="left", pad=10)
    ax.set_xlabel("epoch", fontsize=9, color=MUTED)
    ax.set_ylabel(ylabel, fontsize=9, color=MUTED)
    ax.grid(True, lw=0.6, color="#e6e5e0", zorder=0)      # recessive grid
    ax.set_axisbelow(True)
    for s in ("top","right"): ax.spines[s].set_visible(False)
    for s in ("left","bottom"): ax.spines[s].set_color("#d8d7d1")
    ax.tick_params(colors=MUTED, labelsize=9)

def label_end(ax, x, y, text, color):
    ax.annotate(text, xy=(x[-1], y[-1]), xytext=(4,0), textcoords="offset points",
                color=color, fontsize=9, va="center", fontweight="medium")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
e = df.epoch.values

# 1. losses on ONE axis - train edge loss and validation loss are the same quantity
ax=axes[0]
ax.plot(e, df.train_edge_loss, lw=2, color=BLUE, label="train (edge)", zorder=3)
ax.plot(e, df.val_loss,        lw=2, color=MAGENTA, label="validation", zorder=3)
label_end(ax, e, df.train_edge_loss.values, "train", BLUE)
label_end(ax, e, df.val_loss.values, "val", MAGENTA)
style(ax, "Edge loss — train vs validation", "loss")
ax.legend(frameon=False, fontsize=9, labelcolor=INK)

# 2. validation quality - all rates, so one axis is legitimate
ax=axes[1]
ax.plot(e, df.val_acc,    lw=2, color=BLUE,    label="accuracy", zorder=3)
ax.plot(e, df.val_recall, lw=2, color=GREEN,   label="recall", zorder=3)
ax.plot(e, df.val_acc*df.val_recall, lw=2, color=MAGENTA, ls="--",
        label="score (acc x recall)", zorder=3)
label_end(ax, e, df.val_acc.values, "acc", BLUE)
label_end(ax, e, df.val_recall.values, "recall", GREEN)
style(ax, "Validation quality", "rate")
ax.legend(frameon=False, fontsize=9, labelcolor=INK)
if len(df): ax.axvline(best_ep, color=MUTED, lw=1, ls=":", zorder=1)

# 3. detection loss - different scale, so its own panel rather than a twin axis
ax=axes[2]
ax.plot(e, df.train_det_loss, lw=2, color=BLUE, zorder=3)
label_end(ax, e, df.train_det_loss.values, "det", BLUE)
style(ax, "Detection loss (train)", "loss")

fig.tight_layout()
fig.savefig("/kaggle/working/exp127_curves.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Overfitting diagnosis: compare the trend of train vs validation loss over the
# last third of the run. Divergence (train falling while validation rises) is the
# signature; both still falling means we stopped early, not too late.
import numpy as np
n=len(df)
if n<6:
    print(f"only {n} epochs - too few to judge. Increase TARGET_HOURS or lower MAX_ITERS.")
else:
    k=max(3,n//3); tail=df.tail(k)
    st=np.polyfit(tail.epoch, tail.train_edge_loss, 1)[0]
    sv=np.polyfit(tail.epoch, tail.val_loss, 1)[0]
    print(f"over the last {k} epochs:")
    print(f"  train edge loss slope : {st:+.5f} / epoch")
    print(f"  validation loss slope : {sv:+.5f} / epoch")
    print(f"  best epoch            : {best_ep} of {int(df.epoch.max())}")
    print()
    if st<0 and sv>0:
        print("  OVERFITTING: train loss still falling while validation rises.")
        print("  -> the best checkpoint is the right one to keep; more epochs will not help.")
    elif st<0 and sv<0:
        print("  STILL IMPROVING: both losses falling - this run stopped early, not late.")
        print("  -> more epochs (or a longer budget) should still pay.")
    elif abs(sv)<1e-4:
        print("  PLATEAUED: validation loss flat - extra epochs buy little.")
        print("  -> consider a higher LR, more capacity, or richer augmentation.")
    else:
        print("  UNCLEAR: train loss is not decreasing. Check LR and batch size.")
    frac=best_ep/max(1,int(df.epoch.max()))
    print(f"\n  best epoch is {frac:.0%} of the way through the run"
          + (" - suspiciously early, likely overfitting after it." if frac<0.5 else "."))


In [ ]:
# Persist the trained checkpoint as kernel output so it can become a dataset.
out=Path("/kaggle/working/trained")
out.mkdir(exist_ok=True)
found=list((REPO/"weights").rglob("*.pth"))
for f in found:
    shutil.copy(f, out/f"fold{FOLD}_{f.name}")
    print(f"saved {out/f'fold{FOLD}_{f.name}'}  ({f.stat().st_size/1e6:.1f} MB)")
if not found:
    print("WARNING: no checkpoint written - training may not have completed an improving epoch")
(out/"train_config.json").write_text(json.dumps(
    {"fold":FOLD,"batch":BATCH,"max_iters":MAX_ITERS,"epochs_run":int(df.epoch.max())+1,
     "lr":LR,"best_score":float(df.best.max()),"best_epoch":best_ep,
     "n_train":len(folds[FOLD]["train"]),"n_val":len(folds[FOLD]["test"]),
     "splits_seed":100+FOLD}, indent=2))
for _p in ["submission.csv","/kaggle/working/submission.csv"]:
    assert not os.path.exists(_p)
print("\nconfirmed: no submission.csv; costs no submission slot")
